# MODUL PRAKTIKUM BIG DATA
## Pertemuan 5 — PySpark DataFrame Lanjutan: Multi-DataFrame Join, Window Functions, & Spark SQL (Arch Linux Safe Edition)

| Informasi | Keterangan |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Program Studi** | Teknologi Informasi — Universitas Tidar |
| **Dosen Pengampu** | Alifia Revan Prananda, S.Pd., M.Eng |
| **Sistem Operasi** | Arch Linux / CachyOS (Lingkungan Terisolasi & Aman) |
| **Direktori Kerja** | `/mnt/alterra/repos/praktikum-big-data` (uv project & Git repo) |
| **Perangkat Lunak** | Apache Spark 3.5.9, PySpark 3.5.9, Apache Hadoop 3.4.3, OpenJDK 11 |
| **Topik Pembahasan** | Multi-DataFrame Join, Window Functions (`rank` vs `row_number`), Spark SQL Temporary Views, Arsitektur Hybrid HDFS + Spark |

---

### 🛡️ Jaminan Keamanan Sistem (Zero-Conflict Arch Linux)
Modul versi ini disesuaikan khusus untuk lingkungan **Arch Linux / CachyOS** Anda agar tetap bersih, cepat, dan terisolasi tanpa risiko bentrok dependensi sistem:
1. **Java 11 Terisolasi Penuh**: Spark 3.5.9 berjalan di atas runtime Java 11 (`jdk11-openjdk`) yang dikunci lewat `os.environ["JAVA_HOME"]` dan `spark-env.sh`. Default Java 21 sistem Anda tetap aman dan tidak terganggu.
2. **Lingkungan Portabel (Userspace `~/spark`)**: Seluruh biner Spark ditempatkan di folder `~/spark` milik pengguna. Sama sekali **tidak memerlukan akses root (`sudo`)** untuk operasi rutin.
3. **Manajemen Proyek via `uv`**: Seluruh dependensi Python dijalankan di dalam virtual environment proyek (`.venv`) menggunakan `uv run jupyter notebook`, tanpa menyentuh *system site-packages*.
4. **Variabel Lingkungan HDFS Dinamis (`$USER`)**: Menggunakan variabel pengguna aktif Unix (`$USER`) untuk jalur penyimpanan HDFS (misal `hdfs://localhost:9000/user/kaky/...`), menggantikan jalur *hardcoded* `/user/mahasiswa/...` agar kompatibel langsung dengan izin akses klaster Anda.

---
## 1. Landasan Teori & Prasyarat Ekosistem

### Checklist Sebelum Memulai:
Sebelum menjalankan cell di bawah, pastikan service SSH dan daemon Hadoop dari pertemuan sebelumnya sudah aktif:
- [x] Service SSH aktif: `sudo systemctl start sshd`
- [x] Daemon Hadoop aktif: `start-dfs.sh` dan `start-yarn.sh`
- [x] Verifikasi daemon aktif (`jps`): minimal terdapat 5 proses Java (`NameNode`, `DataNode`, `SecondaryNameNode`, `ResourceManager`, `NodeManager`).
- [x] Lingkungan proyek aktif via `uv` (bukan conda).

---

### Tujuan Pembelajaran:
Setelah menyelesaikan modul Pertemuan 5 ini, Anda diharapkan mampu:
1. **Menggabungkan (*Join*) Multi-DataFrame**: Memahami konsep *relational join* (`inner`, `left`, `right`, `outer`) pada data terdistribusi serta menerapkan *best practice* (agregasi sebelum join).
2. **Menerapkan *Window Function***: Melakukan kalkulasi statistik dan perankingan dalam partisi data tertentu (`rank()` vs `row_number()`) tanpa menghilangkan rincian baris data.
3. **Mengeksekusi *Spark SQL***: Mendaftarkan DataFrame sebagai *temporary view* dan menjalankan kueri SQL deklaratif di atas mesin komputasi Spark (*Catalyst Optimizer*).
4. **Analisis Komparatif**: Menentukan kapan menggunakan DataFrame API murni dan kapan memanfaatkan Spark SQL dalam pipeline rekayasa data.

In [ ]:
import os
import sys
import getpass
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

# 1. Kunci konfigurasi lingkungan terisolasi untuk sub-shell kernel ini
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# 2. Inisialisasi SparkSession dalam mode local[*]
spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL-ArchLinux") \
    .master("local[*]") \
    .getOrCreate()

# 3. Mereduksi pesan log diagnostik agar output notebook bersih dan rapi
spark.sparkContext.setLogLevel("ERROR")

USERNAME = getpass.getuser()
print("=== STATUS KONEKSI SPARK SESSION ===")
print(f"User Host OS : {USERNAME}")
print(f"Versi Spark  : {spark.version}")
print(f"Master Mode  : {spark.sparkContext.master}")
print(f"Aplikasi     : {spark.sparkContext.appName}")
print("Status       : SparkSession aktif & terisolasi penuh di Arch Linux!")

---
## 2. Eksplorasi Skenario Bisnis & Dataset Sintetis

Dalam skenario dunia nyata (*real-world data engineering*), data analitik hampir tidak pernah tersimpan hanya dalam satu tabel tunggal. Data umumnya terdistribusi menjadi:
1. **Tabel Fakta (*Fact Table*) — `df_transaksi`**: Berisi data operasional harian bervolume tinggi yang terus bertambah (misalnya data penjualan dan omzet).
2. **Tabel Dimensi (*Dimension Table / Reference*) — `df_produk`**: Berisi data referensi atau master yang relatif statis dan jarang berubah (misalnya target bulanan dan *Person in Charge* / *Manager* per kategori).

Jalankan cell di bawah ini untuk membangkitkan kedua DataFrame:

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(7)
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]

# Tabel referensi/dimensi: target & manager per kategori (data statis)
data_produk = {
    "kategori": kategori_list,
    "target_bulanan": [50000000, 40000000, 30000000, 25000000, 20000000],
    "manager": ["Andi", "Budi", "Citra", "Dewi", "Eka"],
}
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))

# Tabel fakta/transaksi: data operasional harian
n = 300
data_transaksi = {
    "order_id": [f"O{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))

print("=== Tabel Dimensi Produk (df_produk) ===")
df_produk.show()

print(f"=== Tabel Fakta Transaksi (df_transaksi: 5 baris pertama dari total {df_transaksi.count()} baris) ===")
df_transaksi.show(5)

---
## 3. Operasi Join DataFrame

Sintaks dasar operasi join di PySpark:
```python
df1.join(df2, on="kolom_kunci", how="jenis_join")
```

| Jenis Join | Perilaku & Hasil |
|---|---|
| **`inner`** (default) | Hanya mempertahankan baris yang memiliki nilai kunci yang cocok di **kedua** DataFrame. |
| **`left`** | Mempertahankan seluruh baris dari DataFrame kiri (`df1`), dan melengkapi data dari DataFrame kanan (`df2`) jika ada kecocokan (`null` jika tidak ada). |
| **`right`** | Kebalikan dari `left` — mempertahankan seluruh baris DataFrame kanan. |
| **`outer`** (atau `full`) | Mempertahankan seluruh baris dari kedua DataFrame, baik yang cocok maupun yang tidak. |

---

### 3.1 Left Join — Melengkapi Transaksi dengan Informasi Manager
Left join digunakan ketika kita ingin mempertahankan seluruh 300 transaksi asli, sembari memperkaya kolomnya dengan data nama manager dari `df_produk`:

In [ ]:
# Melakukan Left Join pada kolom 'kategori'
df_gabung = df_transaksi.join(df_produk, on="kategori", how="left")
df_gabung.show(5)

### 3.2 Inner Join Setelah Agregasi (Pola Praktik Terbaik / Best Practice)

Dalam arsitektur Big Data berskala masif, melakukan operasi *join* pada tabel mentah yang memiliki jutaan atau miliaran baris adalah operasi yang mahal karena memicu proses pertukaran data antar-node (*shuffle*).

> 💡 **Pola Terbaik (*Best Practice*):** Lakukan agregasi (*grouping & summary*) terlebih dahulu untuk mereduksi volume data, baru lakukan operasi *join* terhadap tabel referensi.

In [ ]:
# Langkah 1: Ringkas total pendapatan per kategori (mereduksi 300 baris menjadi 5 baris)
ringkasan = df_transaksi.groupBy("kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Langkah 2: Join dengan tabel target untuk mengkalkulasi persentase pencapaian
hasil = ringkasan.join(df_produk, on="kategori", how="inner")
hasil = hasil.withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan") * 100)
)

# Urutkan hasil dari persentase pencapaian tertinggi
hasil.orderBy(col("pencapaian_persen").desc()).show()

Perhatikan pola kerja di atas: **agregasi dulu, baru join** — jauh lebih efisien dibanding join dulu baru agregasi, karena jumlah baris yang di-join menjadi jauh lebih sedikit (5 baris kategori, bukan 300 baris transaksi mentah).

---
## 4. Window Functions di Apache Spark

*Window function* memungkinkan kita melakukan perhitungan agregasi atau perankingan **dalam kelompok (partisi) tertentu tanpa menghilangkan detail baris aslinya**. Ini berbeda mendasar dari `groupBy()` yang mereduksi baris data.

### Skenario Kasus:
Kita ingin mengetahui **transaksi dengan pendapatan tertinggi (top-2)** di **setiap kategori**, sembari tetap mempertahankan atribut detail transaksi (`order_id`, `kota`, `pendapatan`).

In [ ]:
# 1. Definisikan spesifikasi window: partisi berdasarkan kategori, urutkan pendapatan secara descending
window_spec = Window.partitionBy("kategori").orderBy(col("pendapatan").desc())

# 2. Tambahkan kolom peringkat transaksi di dalam kategorinya masing-masing
df_ranked = df_transaksi.withColumn("rank_dalam_kategori", rank().over(window_spec))

# 3. Tampilkan hanya 2 transaksi teratas (rank 1 dan 2) per kategori
df_ranked.filter(col("rank_dalam_kategori") <= 2) \
    .orderBy("kategori", "rank_dalam_kategori") \
    .show(10)

---
### ⚠️ Konsep Fundamental: `rank()` vs `row_number()`
* **`rank()`**: Memberikan nomor peringkat yang sama jika terdapat nilai yang persis sama (*tie*), dan melompati nomor urut berikutnya (misal: 1, 2, 2, 4).
* **`row_number()`**: **Selalu memberikan nomor urut unik dan berurutan** tanpa celah (misal: 1, 2, 3, 4), bahkan jika nilainya kembar. Gunakan `row_number()` jika Anda mewajibkan batasan jumlah baris yang presisi (misalnya wajib tepat 1 transaksi teratas per kelompok).

---
## 5. Spark SQL — Kueri Relasional Deklaratif

Bagi analis dan rekayasawan data yang sudah terbiasa dengan bahasa SQL, Apache Spark menyediakan fitur **Spark SQL**. Dengan Spark SQL, kita dapat menuliskan kueri SQL standar (`SELECT`, `JOIN`, `GROUP BY`, `ORDER BY`) langsung di atas DataFrame.

> ⚡ **Kinerja Identik:** Baik DataFrame API maupun Spark SQL dikompilasi oleh mesin optimasi yang sama (**Catalyst Optimizer** dan **Tungsten Engine**). Keduanya menghasilkan performa komputasi yang setara tanpa penalti kecepatan.

---

**Langkah 1: Daftarkan DataFrame sebagai "tabel sementara" (*temporary view*)**

In [ ]:
# Mendaftarkan temporary view untuk kedua DataFrame
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")

print("Kedua tabel sementara berhasil didaftarkan: 'transaksi' dan 'produk'")

**Langkah 2: Jalankan kueri SQL menggunakan `spark.sql()`**

In [ ]:
hasil_sql = spark.sql('''
    SELECT 
        t.kategori, 
        p.manager, 
        SUM(t.pendapatan) AS total_pendapatan
    FROM transaksi t
    JOIN produk p ON t.kategori = p.kategori
    GROUP BY t.kategori, p.manager
    ORDER BY total_pendapatan DESC
''')

hasil_sql.show()

Perhatikan bahwa hasil kueri SQL di atas **identik** dengan hasil `join()` + `groupBy()` yang kita tulis dengan DataFrame API pada Sub-bab 3 — hanya berbeda gaya penulisan.

### Kapan Memakai DataFrame API, Kapan Memakai Spark SQL?

| Situasi | Rekomendasi |
|---|---|
| Kueri sederhana, tim terbiasa SQL, atau berasal dari sistem lain yang sudah pakai SQL | **Spark SQL** |
| Transformasi kompleks bertahap, butuh reusable function, integrasi erat dengan kode Python lain | **DataFrame API** |
| Ingin fleksibilitas mencampur logika pemrograman (percabangan, perulangan) dengan transformasi data | **DataFrame API** |

Pada praktiknya, banyak data engineer **mencampur keduanya** sesuai kebutuhan — persis seperti yang akan anda lakukan pada Tugas Mandiri minggu ini.

---

## Menutup SparkSession
Sebagai kebiasaan baik dalam rekayasa data, selalu tutup `SparkSession` setelah sesi analisis selesai untuk membebaskan alokasi thread CPU dan memori JVM:

In [ ]:
spark.stop()
print("SparkSession berhasil ditutup. Alokasi thread CPU & memori RAM telah dibebaskan.")

---
## 6. Latihan Mandiri

Jalankan cell di bawah ini untuk membuat sesi `SparkSession` baru khusus latihan serta memuat ulang dataset yang dibutuhkan.

In [ ]:
# Persiapan ulang untuk latihan mandiri
import os
import sys
from pyspark.sql import SparkSession
import numpy as np
import pandas as pd

# Konfigurasi isolasi lingkungan untuk Arch Linux
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("Latihan5-ArchLinux") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

np.random.seed(7)
df_produk = spark.createDataFrame(pd.DataFrame(data_produk))
df_transaksi = spark.createDataFrame(pd.DataFrame(data_transaksi))
df_transaksi.createOrReplaceTempView("transaksi")
df_produk.createOrReplaceTempView("produk")

print("Sesi Spark Latihan siap. DataFrame & Temporary View berhasil dimuat!")

**Soal 1.** Menggunakan **DataFrame API (`join`)**, tampilkan seluruh transaksi kota `"Solo"` beserta nama `manager` dari kategorinya masing-masing.

In [ ]:
# Tuliskan kode jawaban Soal 1 di bawah ini:

**Soal 2.** Menggunakan **window function**, tentukan transaksi dengan `pendapatan` **tertinggi** (peringkat 1 saja) di **setiap kota** (bukan kategori). **Catatan:** Wajib menggunakan `row_number()`, tidak boleh menggunakan `rank()`.

In [ ]:
# Tuliskan kode jawaban Soal 2 di bawah ini:

**Soal 3.** Menggunakan **Spark SQL murni (`spark.sql`)** (bukan DataFrame API), tulis kueri untuk menghitung rata-rata `pendapatan` per `kota`, urutkan dari tertinggi.

In [ ]:
# Tuliskan kode jawaban Soal 3 di bawah ini:

**Soal 4 (Refleksi singkat).** Dalam 2-3 kalimat: menurut anda, dalam situasi seperti apa anda akan lebih memilih menulis Spark SQL dibanding DataFrame API pada pekerjaan anda nanti? Tulis jawaban pada markdown cell di bawah ini.

*(Tuliskan jawaban refleksi Anda pada sel ini)*

---
## 7. TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> 📅 **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 6 dimulai**.  
> 👤 **Sifat tugas:** individu.

### Konteks / Skenario

Manajemen platform e-commerce meminta dibuatkan **dashboard performa cabang toko** yang menggabungkan data transaksi (yang sudah ada di HDFS sejak Pertemuan 3-4) dengan data referensi target penjualan tiap cabang. Anda ditugaskan menyiapkan analisis ini menggunakan kombinasi **join, window function, dan Spark SQL** — persis seperti yang dipelajari hari ini.

### Menyiapkan Dataset

Jalankan cell berikut untuk membuat **dua tabel** dan mengunggah tabel transaksi ke HDFS (tabel target cukup dibuat langsung sebagai Spark DataFrame, karena berukuran kecil dan jarang berubah — praktik umum untuk tabel referensi/*dimension table*).

In [ ]:
import os
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}

df_t5_local = pd.DataFrame(data_transaksi_t5)
df_t5_local.to_csv("transaksi_tugas5.csv", index=False)
print(f"Dataset lokal berhasil dibuat: {df_t5_local.shape[0]} baris.")

# Mengunggah ke klaster HDFS menggunakan variabel dinamis $USER (Arch Linux standard)
!hdfs dfs -mkdir -p /user/$USER/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/$USER/tugas5/
username = os.environ.get("USER", "kaky")
print(f"Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/{username}/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

### Instruksi Pengerjaan

Buat notebook baru **`Tugas5_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu:
1. Baca `transaksi_tugas5.csv` **dari HDFS** (`hdfs://localhost:9000/user/$USER/tugas5/transaksi_tugas5.csv`) menjadi `df_transaksi`, tambahkan kolom `pendapatan` (`unit_terjual x harga_satuan`).
2. Buat `df_target` dari dictionary `data_target_cabang` di atas.

Kerjakan bagian **A sampai D** berikut:

---

**A. Join & Perbandingan Target** *(bobot 25%)*

Ringkas total `pendapatan` per `kota` dari `df_transaksi`, lalu **join** dengan `df_target`. Tambahkan kolom `pencapaian_persen`. Urutkan hasil dari pencapaian tertinggi.

**B. Window Function — Kategori Terlaris per Kota** *(bobot 25%)*

Menggunakan window function, tentukan **kategori dengan pendapatan tertinggi di setiap kota** (top-1 saja, gunakan `row_number()`).

**C. Spark SQL** *(bobot 25%)*

Daftarkan `df_transaksi` dan `df_target` sebagai *temporary view*, lalu **tulis satu kueri SQL** (bukan DataFrame API) yang menampilkan: `kota`, `pic_cabang`, dan jumlah transaksi (`COUNT`) di kota tersebut, diurutkan dari jumlah transaksi terbanyak.

**D. Kesimpulan** *(bobot 25%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan hasil bagian A dan B, **cabang mana yang berkinerja paling baik** dan **cabang mana yang paling perlu perhatian manajemen**? Sertakan angka-angka pendukung dari hasil analisis kalian, bukan opini tanpa dasar data.

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas5_[NPM]_[Nama Lengkap].ipynb` (misal `Tugas5_250506070_Sholahuddin Ahmad.ipynb`) melalui ELITA / Asisten Praktikum, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.
- Bagian A & B **wajib** menggunakan DataFrame API; bagian C **wajib** menggunakan Spark SQL murni (`spark.sql(...)`).

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Join & Pencapaian Target | Join benar, kolom `pencapaian_persen` terhitung tepat | 25% |
| B. Window Function | Kategori terlaris per kota teridentifikasi dengan benar menggunakan `row_number()` | 25% |
| C. Spark SQL | Kueri SQL berjalan benar & menghasilkan output yang sesuai instruksi | 25% |
| D. Kesimpulan | Analisis berbasis data, jelas menyebutkan cabang terbaik & yang perlu perhatian | 25% |